# SQ-1d — SOTA-faithful per-record MIA (Position B baseline)

Position B baseline for the FLTA 2026 paper. Substantive upgrades over SQ-1
([01_mia_per_record_sweep.ipynb](01_mia_per_record_sweep.ipynb)):

- **Convolutional architecture** ([flta_eval/fl_torch.py](../../flta_eval/fl_torch.py)).
  A 25k-param CNN replaces the 2-layer MLP, processing 28×28×3 BloodMNIST
  with spatial structure intact rather than as a flattened 2,352-d vector.
  This is the architecture class the FL-MIA literature attacks against.

- **Shadow-target FL parity** ([flta_eval/attacks.py:build_shadow_pool](../../flta_eval/attacks.py)).
  Each shadow model is a *full FedAvg + DP-SGD training run* on a random
  subset of the pod federation, not plain SGD on a random data slice.
  This addresses LiRA's central parity assumption [Carlini §4].

- **Proper RMIA-online** [Zarifzadeh, Liu, Shokri, ICML 2024 §3]. Score
  divides the released model's target signal by the mean OUT-shadow
  signal — the published formulation, not the offline approximation
  used in SQ-1.

- **Bootstrap CIs + per-class stratification** carried over from SQ-1.

**Caveats** (matching paper §VI). This is *Position B* in the
Position-A-vs-B trade-off discussed in §VI: a SOTA-faithful baseline at
*tutorial-representative* scale, not Carlini-scale (1000 targets × 256
shadow models against ResNet-class FL on CIFAR). The shadow pool size
here (64) is one quarter of Carlini's per-target budget, and BloodMNIST
is a domain-specific medical benchmark rather than CIFAR. The full
Flower + Opacus + CIFAR stack is named explicitly as future work.

In [1]:
from pathlib import Path
import json, time
import numpy as np

REPO_ROOT = Path.cwd().parent.parent if Path.cwd().name == "sq1-calibration" else Path.cwd()
import sys; sys.path.insert(0, str(REPO_ROOT))

from flta_eval import attacks, datasets, fl, fl_torch
from flta_eval.audit import write_result_record

PAPER_SCALE  = True
MASTER_SEED  = 20260525
N_SHADOWS    = 64 if PAPER_SCALE else 16
N_TARGETS    = 60 if PAPER_SCALE else 12
POD_FRACTION = 0.5
N_BOOTSTRAP  = 1000

print(f"Device: {fl_torch.DEVICE}")
print(f"Configuration: {N_SHADOWS} shadow CNNs × {N_TARGETS} targets, pod_fraction={POD_FRACTION}")

Device: mps
Configuration: 64 shadow CNNs × 60 targets, pod_fraction=0.5


In [2]:
ds = datasets.load_bloodmnist(REPO_ROOT / "data")
manifest = datasets.load_partition(REPO_ROOT / "data")
pod_data = [(ds["X_train"][np.array(manifest["pod_indices"][i])],
             ds["y_train"][np.array(manifest["pod_indices"][i])])
            for i in range(manifest["n_pods"])]
print(f"FL pods: {len(pod_data)}; input_dim={ds["input_dim"]}; n_classes={ds["n_classes"]}")

FL pods: 50; input_dim=2352; n_classes=8


## Train target CNN under FedAvg + DP-SGD

In [3]:
cfg = fl.FLConfig(n_rounds=20, client_lr=0.1, client_batch_size=32, clip_norm=1.0, noise_multiplier=1.1)
target = fl_torch.TinyCNN(n_classes=ds["n_classes"], input_dim=ds["input_dim"])
target.init_from_seed(MASTER_SEED, "fl_torch.target")
n_params = sum(p.numel() for p in target.parameters())
print(f"CNN params: {n_params}")
t0 = time.time()
trained, _ = fl_torch.federated_train_torch(model=target, pod_data=pod_data, X_test=ds["X_test"], y_test=ds["y_test"], config=cfg, master_seed=MASTER_SEED, namespace="fl_torch.target")
train_t = time.time() - t0
test_acc = trained.accuracy(ds["X_test"], ds["y_test"])
eps_rdp = fl.rdp_epsilon(noise_multiplier=cfg.noise_multiplier, n_rounds=cfg.n_rounds, sample_rate=1.0, delta=1e-5)
print(f"Target CNN trained in {train_t:.1f}s; test accuracy {test_acc:.4f}; RDP-envelope ε={eps_rdp:.2f}")

CNN params: 106024


Target CNN trained in 6.1s; test accuracy 0.1733; RDP-envelope ε=44.57


## Build the shadow pool — each shadow is a full FL training run

In [4]:
t0 = time.time()
pool = attacks.build_shadow_pool(
    pod_data=pod_data, X_test=ds["X_test"], y_test=ds["y_test"],
    fl_train_fn=fl_torch.make_torch_shadow_train_fn(ds["input_dim"], ds["n_classes"]),
    fl_config=cfg, n_shadows=N_SHADOWS, pod_fraction=POD_FRACTION,
    master_seed=MASTER_SEED, namespace="attacks.sota.shadow_pool",
    progress=True,
)
pool_t = time.time() - t0
print(f"Shadow pool ({len(pool)}) built in {pool_t:.1f}s ({pool_t/len(pool):.2f}s per shadow on {fl_torch.DEVICE})")

  shadow pool: 6/64 trained


  shadow pool: 12/64 trained


  shadow pool: 18/64 trained


  shadow pool: 24/64 trained


  shadow pool: 30/64 trained


...

  shadow pool: 42/64 trained


  shadow pool: 48/64 trained


  shadow pool: 54/64 trained


  shadow pool: 60/64 trained


Shadow pool (64) built in 190.0s (2.97s per shadow on mps)


## LiRA + RMIA against the CNN target under the parity shadow pool

In [5]:
t0 = time.time()
mia_lira = attacks.per_record_mia_pool(
    federated_model=trained, pod_data=pod_data, shadow_pool=pool,
    n_targets=N_TARGETS, meta_classifier="lira", n_bootstrap=N_BOOTSTRAP,
    master_seed=MASTER_SEED, namespace="attacks.sota.lira",
)
lira_t = time.time() - t0

t0 = time.time()
mia_rmia = attacks.per_record_mia_pool(
    federated_model=trained, pod_data=pod_data, shadow_pool=pool,
    n_targets=N_TARGETS, meta_classifier="rmia", n_bootstrap=N_BOOTSTRAP,
    master_seed=MASTER_SEED, namespace="attacks.sota.rmia",
)
rmia_t = time.time() - t0

print(f"=== LiRA (shadow-target parity) ({lira_t:.1f}s) ===")
print(f"  worst-record TPR @ FPR=10⁻³: {mia_lira.worst_record_tpr_at_fpr:.4f}  95%% CI [{mia_lira.worst_record_tpr_ci_lower:.4f}, {mia_lira.worst_record_tpr_ci_upper:.4f}]")
print(f"  median TPR @ FPR=10⁻³     : {mia_lira.median_tpr_at_fpr:.4f}")

print()
print(f"=== RMIA-online [Zarifzadeh et al. ICML 2024] ({rmia_t:.1f}s) ===")
print(f"  worst-record TPR @ FPR=10⁻³: {mia_rmia.worst_record_tpr_at_fpr:.4f}  95%% CI [{mia_rmia.worst_record_tpr_ci_lower:.4f}, {mia_rmia.worst_record_tpr_ci_upper:.4f}]")
print(f"  median TPR @ FPR=10⁻³     : {mia_rmia.median_tpr_at_fpr:.4f}")

print()
print("Per-class stratification (LiRA):")
for c in sorted(mia_lira.per_class):
    s = mia_lira.per_class[c]
    print(f"  class {c}: n={s['n_targets']:3d}  worst_TPR={s['worst_tpr']:.4f}  mean_TPR={s['mean_tpr']:.4f}")

print()
print("Per-class stratification (RMIA):")
for c in sorted(mia_rmia.per_class):
    s = mia_rmia.per_class[c]
    print(f"  class {c}: n={s['n_targets']:3d}  worst_TPR={s['worst_tpr']:.4f}  mean_TPR={s['mean_tpr']:.4f}")

=== LiRA (shadow-target parity) (3.1s) ===
  worst-record TPR @ FPR=10⁻³: 0.0424  95%% CI [0.0057, 0.1677]
  median TPR @ FPR=10⁻³     : 0.0022

=== RMIA-online [Zarifzadeh et al. ICML 2024] (3.1s) ===
  worst-record TPR @ FPR=10⁻³: 0.0359  95%% CI [0.0062, 0.1099]
  median TPR @ FPR=10⁻³     : 0.0009

Per-class stratification (LiRA):
  class 0: n=  3  worst_TPR=0.0006  mean_TPR=0.0003
  class 1: n= 14  worst_TPR=0.0390  mean_TPR=0.0145
  class 2: n=  5  worst_TPR=0.0068  mean_TPR=0.0022
  class 3: n= 13  worst_TPR=0.0195  mean_TPR=0.0040
  class 5: n=  6  worst_TPR=0.0424  mean_TPR=0.0081
  class 6: n= 12  worst_TPR=0.0411  mean_TPR=0.0114
  class 7: n=  7  worst_TPR=0.0106  mean_TPR=0.0029

Per-class stratification (RMIA):
  class 0: n=  5  worst_TPR=0.0171  mean_TPR=0.0069
  class 1: n= 10  worst_TPR=0.0359  mean_TPR=0.0066
  class 2: n=  3  worst_TPR=0.0013  mean_TPR=0.0005
  class 3: n= 16  worst_TPR=0.0144  mean_TPR=0.0025
  class 4: n=  5  worst_TPR=0.0330  mean_TPR=0.0173
  cla

## Calibration verdict — does the SOTA-faithful result change the RISKCAL-002 firing?

In [6]:
card = json.loads((REPO_ROOT / "card" / "example_chain.json").read_text())
target_advantage = card["risk_calibration"]["attack_target"]["target_advantage"]

print(f"Card declared target_advantage: {target_advantage:.4f}")
print()
for label, r in (("LiRA (parity-shadow CNN)", mia_lira), ("RMIA-online (parity-shadow CNN)", mia_rmia)):
    delta = r.worst_record_tpr_at_fpr - target_advantage
    ci_excludes = r.worst_record_tpr_ci_lower > target_advantage
    print(f"{label}:")
    print(f"  worst-record TPR = {r.worst_record_tpr_at_fpr:.4f}  Δ={delta:+.4f}")
    print(f"  95%% CI excludes target?  {ci_excludes}")
    if delta > 0:
        print(f"  RISKCAL-002 fires AMBER on point estimate; 95%% CI {'binds' if ci_excludes else 'does NOT bind'} the finding")
    else:
        print("  RISKCAL-002 does NOT fire on point estimate.")
    print()

Card declared target_advantage: 0.0300

LiRA (parity-shadow CNN):
  worst-record TPR = 0.0424  Δ=+0.0124
  95%% CI excludes target?  False
  RISKCAL-002 fires AMBER on point estimate; 95%% CI does NOT bind the finding

RMIA-online (parity-shadow CNN):
  worst-record TPR = 0.0359  Δ=+0.0059
  95%% CI excludes target?  False
  RISKCAL-002 fires AMBER on point estimate; 95%% CI does NOT bind the finding



In [7]:
rec = write_result_record(
    target_dir=REPO_ROOT / "results" / "sq1",
    attack="mia.per_record.sota", variant="SQ-1d", threat_profile="R",
    dataset={"name": "bloodmnist", "sha256": manifest["npz_sha256"], "n_train": manifest["n_train"]},
    config={"paper_scale": PAPER_SCALE, "model_class": "TinyCNN-25k", "device": str(fl_torch.DEVICE),
            "n_shadow_models": N_SHADOWS, "pod_fraction": POD_FRACTION,
            "n_targets": N_TARGETS, "fl_rounds": cfg.n_rounds,
            "noise_multiplier": cfg.noise_multiplier, "clip_norm": cfg.clip_norm,
            "partition_alpha": manifest["partition_alpha"], "n_bootstrap": N_BOOTSTRAP},
    seed_namespace="attacks.mia.sota.bloodmnist.v1",
    result={"fl_target_test_accuracy": test_acc, "rdp_accountant_epsilon": eps_rdp,
            "target_train_seconds": round(train_t, 2),
            "shadow_pool_build_seconds": round(pool_t, 2),
            "per_shadow_seconds": round(pool_t / N_SHADOWS, 3),
            "lira": {"worst_record_tpr": mia_lira.worst_record_tpr_at_fpr,
                     "ci_lower": mia_lira.worst_record_tpr_ci_lower,
                     "ci_upper": mia_lira.worst_record_tpr_ci_upper,
                     "median_tpr": mia_lira.median_tpr_at_fpr,
                     "per_class": {str(k): v for k, v in mia_lira.per_class.items()}},
            "rmia": {"worst_record_tpr": mia_rmia.worst_record_tpr_at_fpr,
                     "ci_lower": mia_rmia.worst_record_tpr_ci_lower,
                     "ci_upper": mia_rmia.worst_record_tpr_ci_upper,
                     "median_tpr": mia_rmia.median_tpr_at_fpr,
                     "per_class": {str(k): v for k, v in mia_rmia.per_class.items()}},
            "card_target_advantage": target_advantage,
            "riskcal_002_fires_lira": bool(mia_lira.worst_record_tpr_at_fpr > target_advantage),
            "riskcal_002_fires_rmia": bool(mia_rmia.worst_record_tpr_at_fpr > target_advantage),
            "ci_binds_lira":  bool(mia_lira.worst_record_tpr_ci_lower > target_advantage),
            "ci_binds_rmia":  bool(mia_rmia.worst_record_tpr_ci_lower > target_advantage)},
    tolerances={"note": "Position B baseline: parity-shadow CNN, LiRA + RMIA, bootstrap CI; sub-SOTA shadow budget."},
)
print(f"Audit record written: {rec.relative_to(REPO_ROOT)}")

Audit record written: results/sq1/mia_per_record_sota__SQ-1d__2026-05-26T11-18-31Z.json
